# Anderson acceleration using openmc as the transport code 

For now, we just focus on the implicit euler version of this (simpler to test)

SCHEME:
1. DO Starting transport: $\mathrm{IC} = x_0 = \psi^0 \leftarrow T_0$

2. DO Predictor: $P(\psi^0)$
3. SET: $\vec{N}$
4. DO transport: $\psi_1 \leftarrow T$
5. DO Corrector: $C(\psi_1)$
6. SET: $\vec{N}$
7. DO transport: $f(\psi_1) \leftarrow T$
8. BUILD $\vec{x}$: $\vec{x}=\left[\psi^0, \psi_1\right]$
9. BUILD $g$: $g = \left[(\psi_1 - \psi_0), f(\psi_1) - \psi_1  \right]$
10. DO AA: $x_{next} = \mathbf{F(g,x)}$
11. APPEND: $\vec{x}.\mathrm{append}(x_{next})$
12. DO Corrector: $C(\vec{x}_{next})$
13. DO transport: $\psi_2 \leftarrow T$
14. APPEND: $\vec{g}.\mathrm{append}(\psi_2 - x_{next})$
10. DO AA: $x_{next} = \mathbf{F(g,x)}$

In [ ]:
import openmc_aa.data.pwr_rei_template as pwr
import openmc
import openmc.deplete
import numpy as np
import copy
import pickle as pkl
import glob
import os
import matplotlib.pyplot as plt
openmc.deplete.pool.USE_MULTIPROCESSING=False

"""
Depletion scheme for explicit euler. Meant to just get cross sections at the very end for reuse later.
"""

class Anderson():
  """
  Class that contains useful information 
  for doing anderson acceleration depletion
  """
  def __init__(self):
    self._x:     dict[float: list[np.ndarray]] = {}      #  x[time] -> x
    self._fx:    dict[float: list[np.ndarray]] = {}      # fx[time] -> fx
    self._g:     dict[float: list[np.ndarray]] = {}      #  g[time] -> g
    self._k = dict[float: int]
    self._x0: np.ndarray = None

  @property 
  def times(self) -> list[float]:
    return [float(key) for key in self._x.keys()]
  @property
  def x(self) -> dict[float: list[np.ndarray]]:
    return self._x
  @property
  def fx(self) -> dict[float: list[np.ndarray]]:
    return self._fx
  @property
  def g(self) -> dict[float: list[np.ndarray]]:
    return self._g
  @property
  def k(self) -> dict[float: int]:
    return self._k
  @property
  def x0(self) -> np.ndarray:
    return self._x0
  
  def finalize_bos(self, x: np.ndarray):
    """
    Finalizes the BOS results
    
    Parameters
    ==========
    x : np.ndarray
      1d np array for the BOS fluxes (x0)
    """
    t = 0.0
    self._x0 = x
    self._x[t] = [x]
    self._fx[t] = [] # we dont actually solve for fx so dont add anything here.
    self._g[t] = []
    self._k[t] = -1

  def finalize(self, 
               time: float, 
               x: list[np.ndarray], 
               fx: list[np.ndarray], 
               g: list[np.ndarray], 
               k: int):
    """
    Finalizes results after a timestep.
    """
    if time in self.times:
      raise Exception("Time already in self.times, cannot overwrite.")
    self._x[time] = x
    self._fx[time] = fx
    self._g[time] = g
    self._k[time] = k

  def dump_to_pkl(self, name: str):
    """
    Dumps self to a pkl file.

    Parameters
    ==========
    name : str
      name of the file to dump to
    """
    with open(name, "wb") as file:
      pkl.dump(self, file)


  def get_final_tally(self, res: dict, normalize_to: float = 1.0):
    """
    Description
    ===========
    Take in results from a batch-wise transport calculation
    
    Parameters
    ==========
    res : dict 
      results obtained from run_transport()
    normalize_to : float = 1.0
      value to normalize the tally to upon output 
    

    Outputs
    =======
    out : dict
      dictionary of tallies by generation 
    """
    theLength = res[0].__len__()
    shape0 = np.zeros(theLength)
    d = {}
    """nice little function to get the very last tally"""
    maxx = max(list(res.keys()))
    for key in [maxx]: # Cheat code to just get the last generation tall
      shape1 = np.array([ this[:,:,1][0][0] for this in res[key][0:theLength] ])
    shape1 = shape1/np.sum(shape1) * normalize_to
    return shape1
  
  def tally_by_gen(self, res: dict):
    """
    Description
    ===========
    Take in results from a batch-wise transport calculation
    and outputs tallies by generation
    
    Parameters
    ==========
    res : dict 
      results obtained from run_transport()

    Outputs
    =======
    out : dict
      dictionary of tallies by generation 
    """
    theLength = res[0].__len__()
    shape0 = np.zeros(theLength)
    d = {}
    """nice little function to get tallies by gen"""
    for key in res.keys():
      shape1 = np.array([ this[:,:,1][0][0] for this in res[key][0:theLength] ])
      shape = shape1 - shape0
      d[key] = shape
      # advance
      shape0 = shape1
    return d

  def solve(self, 
            x: np.ndarray,
            tidx: int,
            iidx: int, 
            depl_mats: openmc.Materials,
            model: openmc.Model,
            micro_xs: list,
            chain_file: str,
            dt: float,
            power: float,
            depl_id_list: list[int]) -> np.ndarray:
    """
    Solves corrector + transport. Outputs f(x)

    f(x) is the solution to the coupled problem:
      f(x) = Transport(Corrector(x))

    Essentially an evaluation of f(x)
    
    Parameters
    ==========
    x : np.ndarray
      x vector used to deplete (input x)
    tidx : int
      time index
    iidx : int
      iteration index
    depl_mats : openmc.Materials
      materials to be depleted.
    model : openmc.Model
      model object for transport and depletion
    micro_xs : list[openmc.MicroXS]
      list of openmc micro xs
    chain_file : str
      filename of the depletion chain to use in depletion
    dt : float
      delta time (days)
    power : float
      power input to deplete with
    depl_id_list : list[int]
      list of depletion ids for depletion

    Outputs
    =======
    fx : np.ndarray
      output result
    """
    # Name the depletion output
    depl_output_name = f"depl_step_s{tidx+1}_i{iidx}.h5" # PREDICTOR: depl_step_s{TIME_IDX+1}_i{0}.h5 # made to align logically with the transport grid
    
    # Perform depletion until EOS
    depl_flux = copy.deepcopy(x)
    op = openmc.deplete.IndependentOperator(depl_mats, depl_flux, micro_xs, chain_file=chain_file)
    openmc.deplete.PredictorIntegrator(op, timesteps=[dt], power=power, timestep_units='d').integrate(path=depl_output_name)
    make_transport_material_library(output_name=depl_output_name, model=model, chain_file=chain_file)
    
    # Results from transport 
    tr_dict = run_transport(model=model, power_tally_ids=depl_id_list) ## this one for res tracking...
    fx = self.get_final_tally(res=tr_dict, normalize_to=1.0)
    return fx


"""
Regression scheme
"""
class Regression():
  def __init__(self):
    pass

  def get_avg(self, res: dict, val: float):
    """
    gets average from a results dict.
    Note that res must be ran through self.tally_by_gen
    before the averages can be computed.

    Parameters
    ==========
    res : dict
      batch-by-batch estimate of the flux (each batch represents a single trial)
    val : float
      what to normalize the output flux to

    Returns
    =======
    flux : np.array
      the normalized average flux (norm'd across all gens)
    """
    flux = np.zeros(len(res[0]))
    for key in res.keys():
      # Get the normalized flux (this batches estimate for flux)
      the_unnorm_flux = res[key]
      sum_unnorm = np.sum(the_unnorm_flux)
      if sum_unnorm > 0:
        the_normd_flux = the_unnorm_flux/sum_unnorm * val
        flux += the_normd_flux
    flux *= val / np.sum(flux)
    return flux

  def write_res_pkl(self, res: dict, file:str):
    with open(file, 'wb') as f:
      pkl.dump(res, file)

  def tally_by_gen(self, res: dict):
    """does a quick cleanup after running transport"""
    theLength = res[0].__len__()
    shape0 = np.zeros(theLength)
    d = {}
    """nice little function to get tallies by gen"""
    for key in res.keys():
      shape1 = np.array([ this[:,:,1][0][0] for this in res[key][0:theLength] ])
      shape = shape1 - shape0
      d[key] = shape
      # advance
      shape0 = shape1
    return d

  def normalize_res(self, res: dict, val: float):
    """normalizes res dictionary to some value"""
    for key in res.keys():
      the_sum = np.sum(res[key])
      if the_sum == 0:
        res[key] = res[key]
      else:
        res[key] = res[key] / the_sum * val
    return res

  def _get_vij(self, N: int, start: int, F: list, I: list,
              i: int, j: int, f: int):
    Vij = 0.0
    for k in range(start,start+N):
      Vij += (F[i][k][f] - I[i][f])*(F[j][k][f] - I[j][f]) / (N-1)
    return Vij

  def get_new_I(self, N: int, start: int,
                F: list, I: list,
                f: int):
    """
    Parameters
    ==========
    N : int
      number active histories

    start : int
      starting aka nSkipped

    F : list
      list of res dicts (F1, F2, F3, ...)

    I : list
      list of final I values (normalized?)

    f : int
      tally id number to operate on.

    Returns
    =======
    new_I : float
      the new value of I corresponding to tally id f
    coeff : float
      the correlation coefficient

    """
    n = len(F)
    V = np.zeros((n,n))
    for i in range(n):
      for j in range(n):
        V[i,j] = self._get_vij(N=N, start=start, F=F, I=I, i=i, j=j, f=f) # returns vectror of Vij where each index is a fuel zone

    print("the matrix V is")
    print(V)
    if n == 2:
      # print correlation a and b
      coeff = V[1,0]/V[0,0]**0.5/V[1,1]**0.5
      print("the correlation coeff = ", coeff)
    else:
      coeff = 0
    the_I = np.array([[this[f] for this in I]]).transpose() # col vec of my best estimates.
    X = np.ones((n,1)) # col vec of ones.
    M = np.linalg.inv(X.transpose() @ np.linalg.inv(V) @ X) @ X.transpose() @ np.linalg.inv(V)
    print("\nThe matrix is: ", M, "\n")
    new_I = M @ the_I
    return new_I, coeff


"""
Gets the chain (e.g. xs that are pre-tallied)
"""
def run_transport_for_chain(model: openmc.Model, chain_file: str):
  """runs an openmc transport calculation for getting the depletion chain"""
  able_mats = depletable_mats_from_model(model) # get the depletable materials (so we can tally)
  fluxes, micros = openmc.deplete.get_microxs_and_flux(model, able_mats, chain_file=chain_file)
  return fluxes, micros

"""
Batch-by-batch transport simulation in OpenMC
"""
def run_transport(model: openmc.Model, power_tally_ids: list):
  """runs an openmc transport calculation while doing batch-by-patch tallies"""

  # Clear xml's
  for file in glob.glob("*.xml"):
    os.remove(file)

  # TODO DELETE ME for testing stuff
  model.settings.particles = 500
  
  # Export model to XML
  model.export_to_xml()
  
  res = {} # contains/stores power tally ids and stuff like that.

  openmc.lib.init() # initialize
  openmc.lib.simulation_init()
  for b in range(model.settings.batches):
    tallies = [openmc.lib.tallies[the_id] for the_id in power_tally_ids]
    openmc.lib.next_batch()
    results = [tally.results for tally in tallies]
    res[b] = copy.deepcopy(results)
  openmc.lib.simulation_finalize()
  openmc.lib.finalize()
  return res

"""
Gets depletion materials from the Model object 
as an openmc.Materials object
"""
def depletable_mats_from_model(model: openmc.Model) -> openmc.Materials:
  """get depletable materials as openmc.Materials object"""
  depletable_mats = []
  for this in model.materials:
    if this.depletable:
      depletable_mats.append(this)
  depletable_mats = openmc.Materials(depletable_mats)
  return depletable_mats


"""
Disgusting function to get nuclides to 
use/broadcast in transport simulations (addnux functionality basically)
"""
def get_nuclides_for_transport(chain_file: str, model: openmc.Model):
  from openmc.deplete.coupled_operator import _find_cross_sections, _get_nuclides_with_data
  from openmc.deplete.chain import Chain
  chain = Chain.from_xml(chain_file)
  cross_sections = _find_cross_sections(model)
  nuclides_with_data = _get_nuclides_with_data(cross_sections)
  nuclides = [nuc.name for nuc in chain.nuclides
              if nuc.name in nuclides_with_data]
  return nuclides

def make_transport_material_library(output_name: str, model: openmc.Model, chain_file: str):
  """
  Function to take in a model, chain, and reults file.

  Updates the model.materials to be the transport materials
  with the latest results from results file. Inline modification

  Only considers transport nuclides though.
  """

  # Make transport material library.

  results = openmc.deplete.Results(output_name)
  transport_mats = []

  # Depletables
  trans_nuc_list = get_nuclides_for_transport(chain_file=chain_file, model=model)
  for mat in model.materials:
    if not mat.depletable:
      continue # skip if not depletable

    # Make a new material for the depletables
    new_mat = openmc.Material(mat.id, mat.name, temperature=mat.temperature)
    new_mat.volume = mat.volume
    new_mat.depletable = True
    for nuc in trans_nuc_list:
      perc = results.get_atoms(mat=mat, nuc=nuc, nuc_units='atom/b-cm')[-1][-1]
      new_mat.add_nuclide(nuclide=nuc, percent=perc, percent_type='ao')
      new_mat.set_density(units='sum')
    transport_mats.append(new_mat)

  # Nondepletables, can just append what we have already
  for mat in model.materials:
    if not mat.depletable:
      transport_mats.append(mat)

  new_lib = openmc.Materials(transport_mats)
  # new_lib.export_to_xml()
  model.materials = new_lib

def get_depletion_materials_from_results_EOS(output_name: str, model: openmc.Model):
  """
  Function for getting materials for depletion EOS values (or BOS for the next step)
  Returns a list of materials marked depletable
  with full depletion chain.

  No inline modification of models object.
  """
  results = openmc.deplete.Results(output_name)
  depletion_mat_list = []
  for mat in model.materials:
    if mat.depletable:
      eos_mat = results[-1].get_material(str(mat.id))
      depletion_mat_list.append(eos_mat)
  return openmc.Materials(depletion_mat_list)

def chain_from_pkl(file: str):
  with open(file, 'rb') as f:
    fakeFluxes, chain = pkl.load(f)
    if len(chain) == 1:
      new_chain = []
      for this in range(16):
        new_chain.append(copy.deepcopy(chain[0]))
      return new_chain
    else:
      return chain

# Now generating the model

In [ ]:
"""some random settings"""
fuel_r=0.3975
power_density = 104
power = power_density * 366  * np.pi * fuel_r**2
dt = [0.5, 1, 1.5, 2, 5, 10, 20] # up to 20 days (when it gets bad hehe)
iterations = 1 # number of iterations we run (so +1 to this for total # transports)
Nstart = 1
Nactive = 9

"""get the model"""
model = pwr.get_model()
chain_file = '../openmc_RIE/chain_casl_pwr.xml'

"""regression class"""
andy =  Anderson()

"""get depletion materials and a list of depletable materials"""
depletion_materials = depletable_mats_from_model(model=model) # get from starting model
depl_id_list = [this.id for this in depletion_materials]

model.settings

In [ ]:
# Obtain the micro cross sections that we use to deplete (obtained from a separate calculation)
micro_xs = chain_from_pkl(file='../openmc_RIE/chain_gen/FINAL_CHAIN.pkl') # get xs from a reference file

"""
Start
"""
tidx = 0 # timestep index
iidx = 0 # iteration index
this_dt = 0.5 # this delta t

# Run the transport simulation with our model object.
d = run_transport(model=model, power_tally_ids=depl_id_list) ## this one tracks batch-by-batch

# Get the normalized flux shape function from the very last generation/tally 
flux_shape = andy.get_final_tally(res=d, normalize_to=1.0)

# Now deplete forward in time
depl_output_name = f"depl_step_s{tidx}_i{iidx}.h5" # depl_step_s[time_idx]_i[iteration_idx]
op = openmc.deplete.IndependentOperator(depletion_materials, flux_shape, micro_xs, chain_file=chain_file)
openmc.deplete.PredictorIntegrator(op, timesteps=[this_dt], power=power, timestep_units='d').integrate(path=depl_output_name)

# Now update the transport materials (inline modification of model.materials)
make_transport_material_library(output_name=depl_output_name, model=model, chain_file=chain_file)

# Now do transport hehaw-hehaw




# Iterative scheme with anderson acceleration

In [ ]:
# Obtain the micro cross sections that we use to deplete (obtained from a separate calculation)
micro_xs = chain_from_pkl(file='../openmc_RIE/chain_gen/FINAL_CHAIN.pkl') # get xs from a reference file

"""
Input
"""
dt = [0.5, 1, 1.5, 2, 5, 10, 20, 20, 20, 20, 20, 20] # up to 20 days
model = pwr.get_model()
chain_file = '../openmc_RIE/chain_casl_pwr.xml'
depletion_materials = depletable_mats_from_model(model=model) # get from starting model
depl_id_list = [this.id for this in depletion_materials]

# tidx = 0 # timestep index
# iidx = 0 # iteration index
# this_dt = 0.5 # this delta t


"""Start by performing t=0 transport"""
RESULTS_TRANSPORT = run_transport(model=model, power_tally_ids=depl_id_list) ## transport w/ batch-by-batch tally tracking
LATEST_FLUX = andy.get_final_tally(res=RESULTS_TRANSPORT, normalize_to=1.0)

for TIME_IDX, this_dt in enumerate(dt):
  """Construct x and g"""
  ITERATION_IDX: int = int(0)
  x = [copy.deepcopy(LATEST_FLUX)] # construct x using initial conditions
  g = []
  

  """Start by doing predictor depletion using latest flux"""
  depl_output_name = f"depl_step_s{TIME_IDX+1}_i{ITERATION_IDX}.h5" # PREDICTOR: depl_step_s{TIME_IDX+1}_i{0}.h5 # made to align logically with the transport grid
  op = openmc.deplete.IndependentOperator(depletion_materials, LATEST_FLUX, micro_xs, chain_file=chain_file)
  openmc.deplete.PredictorIntegrator(op, timesteps=[this_dt], power=power, timestep_units='d').integrate(path=depl_output_name)
  
  """SET the nuclide densities at EOS"""
  make_transport_material_library(output_name=depl_output_name, model=model, chain_file=chain_file)

  """DO transport at EOS to obtain psi_1 (first transport solve)"""
  RESULTS_TRANSPORT = run_transport(model=model, power_tally_ids=depl_id_list) ## this one for res tracking...
  LATEST_FLUX = andy.get_final_tally(res=RESULTS_TRANSPORT, normalize_to=1.0)
  
  """Update x and g according to results"""
  x.append(copy.deepcopy(LATEST_FLUX))
  g.append(x[1]- x[0])

  """SOLVE again to obtain g[x1]"""


  




# version with solve() as a function so that the scheme is much mroe clearer/

In [ ]:
"""
Input
"""
# Input stuff
micro_xs = chain_from_pkl(file='../openmc_RIE/chain_gen/FINAL_CHAIN.pkl') # get xs from a reference file
dt = [0.5, 1, 1.5, 2, 5, 10, 20, 20, 20, 20, 20, 20] # up to 20 days
model = pwr.get_model()
chain_file = '../openmc_RIE/chain_casl_pwr.xml'

# Computing the power density to use
fuel_r=0.3975
power_density = 104
power = power_density * 366  * np.pi * fuel_r**2

# Random stuff - non-strictly-input-based
depletion_materials = depletable_mats_from_model(model=model) # get from starting model
depl_id_list = [this.id for this in depletion_materials]

# AA related
mr = 4 # number of past iterations to use in AA
max_solves = 3
tol_res = 1e-6
aa = Anderson()


"""
BEGIN DEPLETION SCHEME
"""

"""Start by performing t=0 transport"""
RESULTS_TRANSPORT = run_transport(model=model, power_tally_ids=depl_id_list) ## transport w/ batch-by-batch tally tracking
LATEST_FLUX = aa.get_final_tally(res=RESULTS_TRANSPORT, normalize_to=1.0)
aa.finalize_bos(x=LATEST_FLUX)

the_eos_time = 0.0
for TIME_IDX, this_dt in enumerate(dt):
  # Time
  the_eos_time += this_dt
  
  # Set x and g and f(x). 
  x =  [copy.deepcopy(LATEST_FLUX)] # construct x using initial conditions
  fx = []
  g =  [] # construct g at this timestep.

  # Solve f(x1)
  fx1 = aa.solve(x = LATEST_FLUX, 
                 tidx=TIME_IDX, 
                 iidx=1,  # the resulting index of x (this will be x[1] upon solving)
                 depl_mats=depletion_materials,
                 model=model, micro_xs=micro_xs,chain_file=chain_file,
                 dt=this_dt, power=power, depl_id_list=depl_id_list)
        
  x.append(copy.deepcopy(fx1))
  fx.append(copy.deepcopy(fx1))
  g.append(x[1]-x[0])

  # Solve f(x2)
  fx2 = aa.solve(x = fx1, 
                 tidx=TIME_IDX, 
                 iidx=2,  # the resulting index of x (this will be x[2] upon solving)
                 depl_mats=depletion_materials,
                 model=model, micro_xs=micro_xs,chain_file=chain_file,
                 dt=this_dt, power=power, depl_id_list=depl_id_list)
  fx.append(copy.deepcopy(fx2))
  g.append(fx[1] - fx[0])

  # Matrices G_k ad X_k
  d = LATEST_FLUX.size
  G_k = (g[1] - g[0]).reshape(d, 1)
  X_k = (x[1] - x[0]).reshape(d, 1)
  
  keepGoing, k = True, int(2)
  while keepGoing:
    m_k = min(k,mr)

    # Solve least squares: min || G_k gamma - g_k ||_2
    Q, R = np.linalg.qr(G_k, mode='reduced')      # Q:(d,p), R:(p,p)
    rhs = Q.T @ g[k-1].reshape(d, 1)              # (p,1)
    gamma_k = np.linalg.lstsq(R, rhs, rcond=None)[0]  # (p,1)

    # Get intermediate x_next
    x_next = x[k-1] + g[k-1] - ((X_k + G_k) @ gamma_k).reshape(d)

    fx_next = aa.solve(x = x_next, 
                       tidx=TIME_IDX, 
                       iidx=k+1,  # the resulting index of x (this will be x[1] upon solving)
                       depl_mats=depletion_materials,
                       model=model, micro_xs=micro_xs,chain_file=chain_file,
                       dt=this_dt, power=power, depl_id_list=depl_id_list)
    g_next = fx_next - x_next

    # NOTE: fx gets fx_next (f(n_next)) and x gets x_next (so they are NOT the same after all)
    x.append(x_next)
    fx.append(fx_next)
    g.append(g_next)

    # Update increment matrices
    X_k = np.hstack([X_k, (x[k] - x[k-1]).reshape(d, 1)])
    G_k = np.hstack([G_k, (g[k] - g[k-1]).reshape(d, 1)])

    # Keep only last m_k columns
    ncols = X_k.shape[1]
    if ncols > m_k:
      X_k = X_k[:, ncols - m_k:]
      G_k = G_k[:, ncols - m_k:]

    # Convergence criteria
    nrm = np.linalg.norm(g[k], ord=2)
    if (abs(nrm) < tol_res) | (k > max_solves):
      keepGoing = False
      aa.finalize(time=the_eos_time, x=x, fx=fx, g=g, k=k)


    k += 1

# I feel like we should really be breaking after x_next ? idk.



In [ ]:
LATEST_FLUX.size

In [ ]:
d = LATEST_FLUX.size
G_k = (g[1] - g[0]).reshape(d, 1)
X_k = (x[1] - x[0]).reshape(d, 1)

In [ ]:
X_k